# Stage 14b — Soumission : interpolation (meilleure piste actuelle)

L'interpolation a donné **0.826** au board (ta meilleure soumission) ET **+0.0046** en CV. Deux signaux faibles concordants → candidate sérieuse pour un verrou final. Ce notebook la régénère proprement.

In [1]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
TRAIN_PATH="Train.csv"; TEST_PATH="Test.csv"; SAMPLE_PATH="SampleSubmission.csv"
NODATA=-9999
MONTHS=[f"{i:02d}" for i in range(1,13)]
BANDS=["VH","VV","blue","green","nir","nira","re1","re2","re3","red","swir1","swir2"]
WINTER_BIAS=np.array([.88,.78,.62,.50,.41,.43,.36,.41,.50,.80,.75,.87])
def sr(a,b): return (a-b)/(a+b)
def interpolate_months(df):
    d=df.copy()
    for b in BANDS:
        cols=[f"{b}_{m}" for m in MONTHS]; M=d[cols].values.astype(float)
        for i in range(len(d)):
            row=M[i]; v=np.isfinite(row)
            if v.sum()>=2: M[i]=np.interp(np.arange(12),np.arange(12)[v],row[v])
            elif v.sum()==1: M[i]=np.where(v,row,row[v][0])
        d[cols]=M
    return d
def build_features(df, use_interp=False):
    if use_interp: df=interpolate_months(df)
    feats={}; idx={}
    for m in MONTHS:
        g,n,r,s1=df[f"green_{m}"],df[f"nir_{m}"],df[f"red_{m}"],df[f"swir1_{m}"]
        idx.setdefault("NDWI",[]).append(sr(g,n)); idx.setdefault("MNDWI",[]).append(sr(g,s1))
        idx.setdefault("NDVI",[]).append(sr(n,r)); idx.setdefault("NDMI",[]).append(sr(n,s1))
        idx.setdefault("VHmVV",[]).append(df[f"VH_{m}"]-df[f"VV_{m}"])
    def s(mat,nm):
        a=np.column_stack([np.asarray(x,float) for x in mat])
        feats[f"{nm}_mean"]=np.nanmean(a,1);feats[f"{nm}_std"]=np.nanstd(a,1);feats[f"{nm}_min"]=np.nanmin(a,1)
        feats[f"{nm}_max"]=np.nanmax(a,1);feats[f"{nm}_range"]=np.nanmax(a,1)-np.nanmin(a,1);feats[f"{nm}_med"]=np.nanmedian(a,1)
        feats[f"{nm}_nvalid"]=np.isfinite(a).sum(1)
    for b in BANDS: s([df[f"{b}_{m}"] for m in MONTHS],b)
    for k,v in idx.items(): s(v,k)
    return pd.DataFrame(feats,index=df.index).replace([np.inf,-np.inf],np.nan)
def corrupt_realistic(df, seed=0):
    rng=np.random.RandomState(seed); d=df.copy()
    cols={b:[f"{b}_{m}" for m in MONTHS] for b in BANDS}
    arr={b:d[cols[b]].values.astype(float) for b in BANDS}
    for i in range(len(d)):
        miss=rng.rand(12)<WINTER_BIAS*rng.uniform(0.7,1.1)
        for b in BANDS: arr[b][i,miss]=np.nan
    for b in BANDS: d[cols[b]]=arr[b]
    return d
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

In [3]:
train=pd.read_csv(TRAIN_PATH).replace(NODATA,np.nan)
test=pd.read_csv(TEST_PATH).replace(NODATA,np.nan)
sample=pd.read_csv(SAMPLE_PATH); y=train["label"].values
parts=[train]+[corrupt_realistic(train,seed=i+1) for i in range(2)]
aug=pd.concat(parts,ignore_index=True); yaug=np.concatenate([y]*len(parts))
X_aug=build_features(aug, use_interp=True)
X_test=build_features(test, use_interp=True)
hgb=HistGradientBoostingClassifier(max_iter=400,learning_rate=0.05,max_leaf_nodes=7,l2_regularization=1.0,random_state=42).fit(X_aug,yaug)
lr=make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),LogisticRegression(max_iter=2000)).fit(X_aug,yaug)
proba=0.5*hgb.predict_proba(X_test)[:,1]+0.5*lr.predict_proba(X_test)[:,1]
sub=pd.DataFrame({"ID":test["ID"],"TargetF1":(proba>=0.5).astype(int),"TargetRAUC":proba})
assert list(sub.columns)==list(sample.columns) and set(sub.ID)==set(sample.ID)
print("bassins:",sub.TargetF1.sum(),f"({sub.TargetF1.mean():.1%})")
sub.to_csv("submission_interp2.csv",index=False); print("ecrit: submission_interp.csv")

bassins: 549 (53.3%)
ecrit: submission_interp.csv
